In [2]:
"""
Context RAG + History-as-Context (dual retrieval)

What changed vs your original script:
1) We STILL use chat history for pronoun-resolution in the rewrite step.
2) We NOW ALSO treat your CSV Q/A history as *retrievable knowledge* by building a second FAISS
   vector store from the history CSV (Question+Answer).
3) At query time we retrieve from:
     - PDF/vector_db FAISS (your existing INDEX_DIR)
     - History FAISS (built at runtime from HISTORY_CSV_PATH)
   Then we merge both into a single CONTEXT block for the answering prompt.
4) We keep strict answering: the model must answer ONLY from the provided combined context.

Notes:
- This is the simplest “history as context” pattern that still scales and avoids dumping the
  entire history into the prompt.
- History retrieval is configurable (HIST_TOP_K / HIST_FETCH_K / HIST_USE_MMR, etc.)
"""

import os
import re
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()

import faiss  # ✅ needed to read index dimension

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document


# ============================================================
# CONFIG
# ============================================================
HISTORY_CSV_PATH = r"C:\Users\surya.adatravu\Documents\ContextRAG\RA_FSM_QA.csv"
TEST_CSV_PATH    = r"C:\Users\surya.adatravu\Documents\ContextRAG\RA_FSM_RAG_Test_Questions.csv"

INDEX_DIR        = r"C:\Users\surya.adatravu\Documents\ContextRAG\vector_db"
SESSION_ID       = "pdf_faiss_session"
RESULTS_OUT_CSV  = r"C:\Users\surya.adatravu\Documents\ContextRAG\context_rag_test_results_4omini.csv"

# Retrieval (PDF/vector_db)
USE_MMR = True
TOP_K = 6
FETCH_K = 40
LAMBDA_MULT = 0.3
USE_DISTANCE_GATE = True
DISTANCE_THRESHOLD = 0.85

# Retrieval (History store)
USE_HISTORY_AS_CONTEXT = True
HIST_USE_MMR = True
HIST_TOP_K = 4
HIST_FETCH_K = 30
HIST_LAMBDA_MULT = 0.35
HIST_USE_DISTANCE_GATE = False  # often history snippets are “close enough”; gate optional
HIST_DISTANCE_THRESHOLD = 0.95

# Evaluation
VALIDATION_THRESHOLD = 0.5
NORMALIZE_FOR_SCORING = True

# LLM (embedding model will be auto-selected to match PDF FAISS index)
LLM_MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)


# ============================================================
# MESSAGE HISTORY STORE (used for pronoun resolution)
# ============================================================
store = {}

def get_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


def preload_history_from_csv(session_id: str, csv_path: str, max_rows=None, clear_existing=True):
    df = pd.read_csv(csv_path)
    if not {"Question", "Answer"}.issubset(df.columns):
        raise ValueError(f"History CSV must contain columns Question, Answer. Found: {list(df.columns)}")

    history = get_history(session_id)
    if clear_existing:
        history.clear()

    loaded = 0
    for _, row in df.iterrows():
        if max_rows is not None and loaded >= max_rows:
            break
        history.add_user_message(str(row["Question"]))
        history.add_ai_message(str(row["Answer"]))
        loaded += 1

    print(f"✅ Loaded {loaded} Q/A pairs into message history (messages={len(history.messages)}) for session '{session_id}'")
    return history


def load_test_questions(csv_path: str):
    df = pd.read_csv(csv_path)

    possible_q = ["Question", "question", "Query", "query"]
    possible_a = ["Answer", "answer", "Expected", "expected_answer", "GroundTruth", "ground_truth"]

    q_col = next((c for c in possible_q if c in df.columns), None)
    a_col = next((c for c in possible_a if c in df.columns), None)

    if q_col is None:
        raise ValueError(f"Test CSV must include a question column (one of {possible_q}). Found: {list(df.columns)}")
    if a_col is None:
        raise ValueError(f"Test CSV must include an answer column (one of {possible_a}). Found: {list(df.columns)}")

    df = df.rename(columns={q_col: "Question", a_col: "ExpectedAnswer"})
    return df[["Question", "ExpectedAnswer"]]


# ============================================================
# ✅ LOAD PDF FAISS + AUTO-SELECT EMBEDDINGS TO MATCH INDEX DIM
# ============================================================
def embedding_model_for_faiss_dim(d: int) -> str:
    # Common OpenAI embedding dims:
    # text-embedding-3-small -> 1536
    # text-embedding-3-large -> 3072
    if d == 1536:
        return "text-embedding-3-small"
    if d == 3072:
        return "text-embedding-3-large"
    raise ValueError(
        f"Unknown FAISS index dimension d={d}. "
        "Cannot auto-select embedding model. Rebuild index or specify a compatible embeddings model."
    )

def load_faiss_and_embeddings(index_dir: str):
    faiss_path = os.path.join(index_dir, "index.faiss")
    pkl_path   = os.path.join(index_dir, "index.pkl")

    if not (os.path.exists(faiss_path) and os.path.exists(pkl_path)):
        raise FileNotFoundError(f"FAISS index not found in {index_dir}. Build it first.")

    idx = faiss.read_index(faiss_path)
    d = idx.d

    model_name = embedding_model_for_faiss_dim(d)
    print(f"✅ Detected PDF FAISS dim={d}. Using embeddings model: {model_name}")

    embeddings = OpenAIEmbeddings(model=model_name)
    vs = FAISS.load_local(index_dir, embeddings, allow_dangerous_deserialization=True)
    return vs, embeddings, model_name


# ============================================================
# ✅ NEW: Build a History VectorStore (Q/A) using SAME embeddings
# ============================================================
def build_history_vectorstore(history_csv_path: str, embeddings: OpenAIEmbeddings) -> FAISS:
    df = pd.read_csv(history_csv_path)
    if not {"Question", "Answer"}.issubset(df.columns):
        raise ValueError(f"History CSV must contain columns Question, Answer. Found: {list(df.columns)}")

    docs = []
    for i, row in df.iterrows():
        q = str(row["Question"])
        a = str(row["Answer"])

        # Store both Q and A in the document so it can be used as context verbatim.
        content = f"QUESTION:\n{q}\n\nANSWER:\n{a}"

        docs.append(
            Document(
                page_content=content,
                metadata={
                    "source_type": "history_csv",
                    "history_row": int(i),
                    "history_question": q[:200],  # short preview
                },
            )
        )

    print(f"✅ Building History FAISS from {len(docs)} Q/A docs...")
    hist_vs = FAISS.from_documents(docs, embeddings)
    return hist_vs


# ============================================================
# PROMPTS
# ============================================================
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Rewrite the user's latest question into a standalone search query.\n"
     "Use chat history only to resolve pronouns and references.\n"
     "Return ONLY the rewritten query."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# ✅ Updated: history snippets are now part of CONTEXT (knowledge), so allow using them.
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a strict knowledge assistant.\n"
     "You must ONLY answer using the provided CONTEXT.\n"
     "IMPORTANT STYLE RULES:\n"
     "- Use exact wording from CONTEXT whenever possible.\n"
     "- Do NOT add extra explanation or commentary.\n"
     "- If the context contains enumerations/lists/levels/states, reproduce them with same labels/order.\n"
     "- Keep the answer concise.\n"
     "If the answer is not in context, say exactly:\n"
     "\"I don't know from provided knowledge.\""),
    ("human",
     "CONTEXT:\n{context}\n\n"
     "QUESTION:\n{question}\n\n"
     "Answer using ONLY the context (follow style rules).")
])

rewrite_chain = rewrite_prompt | llm
rag_chain = rag_prompt | llm

rewrite_with_history = RunnableWithMessageHistory(
    rewrite_chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history",
)


# ============================================================
# SCORING / NORMALIZATION
# ============================================================
def normalize_for_scoring(s: str) -> str:
    if not isinstance(s, str):
        s = str(s)
    s = s.replace("I don't know from provided knowledge.", "").strip()
    s = re.sub(r"^\s*\[file=.*?\]\s*$", "", s, flags=re.MULTILINE)
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


def similarity_score(text1: str, text2: str, embeddings: OpenAIEmbeddings) -> float:
    v1 = embeddings.embed_query(text1)
    v2 = embeddings.embed_query(text2)
    return float(cosine_similarity([v1], [v2])[0][0])


def validate_pred_vs_expected(pred: str, expected: str, embeddings: OpenAIEmbeddings, threshold: float = VALIDATION_THRESHOLD):
    if NORMALIZE_FOR_SCORING:
        pred = normalize_for_scoring(pred)
        expected = normalize_for_scoring(expected)

    score = similarity_score(pred, expected, embeddings)
    return score >= threshold, score


# ============================================================
# RETRIEVAL HELPERS
# ============================================================
def retrieve_docs(vectorstore: FAISS, query: str, use_mmr: bool, top_k: int, fetch_k: int, lambda_mult: float):
    if use_mmr:
        docs = vectorstore.max_marginal_relevance_search(
            query, k=top_k, fetch_k=fetch_k, lambda_mult=lambda_mult
        )
        scored = vectorstore.similarity_search_with_score(query, k=1)
        top_distance = float(scored[0][1]) if scored else None
        return docs, top_distance
    else:
        docs_with_scores = vectorstore.similarity_search_with_score(query, k=top_k)
        docs = [d for d, _ in docs_with_scores]
        top_distance = float(docs_with_scores[0][1]) if docs_with_scores else None
        return docs, top_distance


def format_docs_as_context(docs, kind: str) -> str:
    """
    kind: "pdf" or "history"
    """
    blocks = []
    if kind == "pdf":
        for d in docs:
            blocks.append(
                f"[file={d.metadata.get('source_file','?')} page={d.metadata.get('page','?')}]\n{d.page_content}"
            )
    elif kind == "history":
        for d in docs:
            blocks.append(
                f"[history row={d.metadata.get('history_row','?')}]\n{d.page_content}"
            )
    else:
        for d in docs:
            blocks.append(d.page_content)
    return "\n\n---\n\n".join(blocks)


def collect_sources(docs, kind: str):
    if kind == "pdf":
        return sorted({
            f"{d.metadata.get('source_file', '?')}#page={d.metadata.get('page', '?')}"
            for d in docs
        })
    if kind == "history":
        return sorted({
            f"history_csv#row={d.metadata.get('history_row','?')}"
            for d in docs
        })
    return []


# ============================================================
# ASK (dual retrieval + combined context)
# ============================================================
def ask(question: str, pdf_vs: FAISS, hist_vs: FAISS | None):
    rewritten = rewrite_with_history.invoke(
        {"input": question},
        config={"configurable": {"session_id": SESSION_ID}}
    ).content.strip()

    # 1) Retrieve from PDF/vector_db
    pdf_docs, pdf_top_dist = retrieve_docs(
        pdf_vs, rewritten, USE_MMR, TOP_K, FETCH_K, LAMBDA_MULT
    )

    if USE_DISTANCE_GATE and pdf_top_dist is not None and pdf_top_dist > DISTANCE_THRESHOLD:
        pdf_docs = []  # treat as no usable pdf context

    # 2) Retrieve from History store (optional)
    hist_docs, hist_top_dist = [], None
    if USE_HISTORY_AS_CONTEXT and hist_vs is not None:
        hist_docs, hist_top_dist = retrieve_docs(
            hist_vs, rewritten, HIST_USE_MMR, HIST_TOP_K, HIST_FETCH_K, HIST_LAMBDA_MULT
        )
        if HIST_USE_DISTANCE_GATE and hist_top_dist is not None and hist_top_dist > HIST_DISTANCE_THRESHOLD:
            hist_docs = []

    # 3) If nothing retrieved anywhere, return IDK
    if not pdf_docs and not hist_docs:
        return {
            "question": question,
            "rewritten_query": rewritten,
            "answer": "I don't know from provided knowledge.",
            "pdf_retrieval_distance": pdf_top_dist,
            "history_retrieval_distance": hist_top_dist,
            "used_sources": "",
        }

    # 4) Build combined context (history first is often helpful, but you can swap)
    parts = []
    if hist_docs:
        parts.append("HISTORY CONTEXT:\n" + format_docs_as_context(hist_docs, "history"))
    if pdf_docs:
        parts.append("DOCUMENT CONTEXT:\n" + format_docs_as_context(pdf_docs, "pdf"))
    context = "\n\n====================\n\n".join(parts)

    # 5) Answer strictly from combined context
    resp = rag_chain.invoke({"question": question, "context": context})

    used_sources = collect_sources(hist_docs, "history") + collect_sources(pdf_docs, "pdf")

    return {
        "question": question,
        "rewritten_query": rewritten,
        "answer": resp.content.strip(),
        "pdf_retrieval_distance": pdf_top_dist,
        "history_retrieval_distance": hist_top_dist,
        "used_sources": " | ".join(used_sources),
    }


def run_tests(pdf_vs: FAISS, hist_vs: FAISS | None, test_df: pd.DataFrame, embeddings: OpenAIEmbeddings):
    rows = []
    pass_count = 0

    for i, r in test_df.iterrows():
        q = str(r["Question"])
        expected = str(r["ExpectedAnswer"])

        out = ask(q, pdf_vs, hist_vs)
        pred = out["answer"]

        ok, sim = validate_pred_vs_expected(pred, expected, embeddings)

        rows.append({
            "idx": i,
            "question": q,
            "expected_answer": expected,
            "predicted_answer": pred,
            "cosine_similarity": sim,
            "pass": ok,
            "rewritten_query": out["rewritten_query"],
            "pdf_retrieval_distance": out["pdf_retrieval_distance"],
            "history_retrieval_distance": out["history_retrieval_distance"],
            "used_sources": out["used_sources"],
        })

        pass_count += int(ok)
        print(
            f"[{i+1}/{len(test_df)}] pass={ok} sim={sim:.3f} "
            f"pdf_dist={out['pdf_retrieval_distance']} hist_dist={out['history_retrieval_distance']}"
        )

    results_df = pd.DataFrame(rows)
    accuracy = pass_count / max(len(test_df), 1)

    print("\n====================")
    print(f"✅ Tests completed: {len(test_df)}")
    print(f"✅ Pass count     : {pass_count}")
    print(f"✅ Accuracy       : {accuracy:.3%}")
    print("====================\n")

    return results_df


# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    # Keep message history preload ONLY for rewrite/pronoun resolution (chat-history semantics)
    preload_history_from_csv(SESSION_ID, HISTORY_CSV_PATH, max_rows=None, clear_existing=True)

    # Load PDF/vector_db store + embeddings (embeddings must match FAISS dim)
    pdf_vs, emb, emb_model_name = load_faiss_and_embeddings(INDEX_DIR)

    # Build a separate vector store for history-as-context, using SAME embeddings
    hist_vs = None
    if USE_HISTORY_AS_CONTEXT:
        hist_vs = build_history_vectorstore(HISTORY_CSV_PATH, emb)

    test_df = load_test_questions(TEST_CSV_PATH)
    print(f"✅ Loaded {len(test_df)} test questions from: {TEST_CSV_PATH}")

    results = run_tests(pdf_vs, hist_vs, test_df, emb)

    os.makedirs(os.path.dirname(RESULTS_OUT_CSV), exist_ok=True)
    results.to_csv(RESULTS_OUT_CSV, index=False)
    print(f"✅ Saved results to: {RESULTS_OUT_CSV}")


✅ Loaded 50 Q/A pairs into message history (messages=100) for session 'pdf_faiss_session'
✅ Detected PDF FAISS dim=1536. Using embeddings model: text-embedding-3-small
✅ Building History FAISS from 50 Q/A docs...
✅ Loaded 25 test questions from: C:\Users\surya.adatravu\Documents\ContextRAG\RA_FSM_RAG_Test_Questions.csv
[1/25] pass=True sim=0.563 pdf_dist=1.0582702159881592 hist_dist=0.8528356552124023
[2/25] pass=True sim=0.617 pdf_dist=0.9055246114730835 hist_dist=0.7375244498252869
[3/25] pass=False sim=0.176 pdf_dist=0.9286885261535645 hist_dist=0.535443127155304
[4/25] pass=False sim=0.175 pdf_dist=1.0011959075927734 hist_dist=1.242555022239685
[5/25] pass=False sim=0.119 pdf_dist=1.2967432737350464 hist_dist=0.891120433807373
[6/25] pass=False sim=0.250 pdf_dist=1.3182315826416016 hist_dist=1.02053964138031
[7/25] pass=False sim=0.130 pdf_dist=0.8966614007949829 hist_dist=0.9517372250556946
[8/25] pass=False sim=0.118 pdf_dist=1.095984935760498 hist_dist=1.04436457157135
[9/25] pa